# V029 results: better individual predictions; fixed mixtures miss the practical gate

Native individual A MAE improved0.20382954→0.20013019. The better of two frozen mixtures scored0.18018912 vs0.18292209, gain0.00273298 below the unchanged0.003 minimum. No F confirmation, final fit or submission follows V029. The official0.17427 result is preserved; <=0.15 remains unmet.

The completed200-epoch run used the same original grouped A split, features, seed, mean body and rate schedule. Only the positive-scale likelihood/head package changed. No checkpoint was selected using NLL or training error. The first unfamiliar objective finished as required; no production pruning occurred. Independent1008 checks passed. These A results are exploratory after repeated research.


In [1]:
from pathlib import Path
import json, hashlib, os
import numpy as np, pandas as pd
P=Path.cwd(); O=P/'outputs/controlled_experiments_v029'; V=O/'review'; V.mkdir(exist_ok=True)
os.environ['MPLCONFIGDIR']=str(V/'mplconfig')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
read=lambda p:json.loads(Path(p).read_text())
r=read(O/'screen_summary.json');audit=read(P/'research/screen_review_v029.json')
assert audit['status']=='passed' and all(audit['checks'].values()) and audit['selected'] is None
assert hashlib.sha256((O/'screen_summary.json').read_bytes()).hexdigest()==audit['screen_summary_sha256']
new=read(O/'laplace_curve/report.json');old=read(P/'outputs/controlled_experiments_v013/deep_curve/report.json')
print('Individual control:',old['validation_mae'],'Laplace:',new['validation_mae'],'selected epoch:',new['best_epoch'])
print(pd.DataFrame([{k:c[k] for k in ['name','MAE','gain','eligible']} for c in r['candidates']]).to_string(index=False))
print('Runtime:',new['elapsed_seconds']/3600,'GPU hours; independent checks:',audit['check_count'])


Individual control: 0.2038295445051591 Laplace: 0.20013018541768796 selected epoch: 196
                  name      MAE     gain  eligible
   laplace_replacement 0.181065 0.001857     False
laplace_retained_curve 0.180189 0.002733     False
Runtime: 2.3746926041264347 GPU hours; independent checks: 1008


In [2]:
df=pd.read_csv(P/'data/train.csv',usecols=['id','breath_id','R','C','u_out','pressure'],dtype={'id':'int64','breath_id':'int32','R':'int8','C':'int8','u_out':'int8','pressure':'float32'})
bids=df.breath_id.to_numpy()[::80];rc=df[['R','C']].to_numpy()[::80];y=df.pressure.to_numpy().reshape(-1,80);phase=df.u_out.to_numpy().reshape(-1,80)
with np.load(P/'outputs/improvements_v005/sequence/split_breath_ids.npz') as z:val=np.sort(pd.Index(bids).get_indexer(z['A_validation']))
mask=phase[val]==0;truth=y[val][mask].astype(np.float64);rows=df.id.to_numpy().reshape(-1,80)[val][mask]
assert np.array_equal(rows,np.load(O/'A_row_ids.npy'))
mu=np.load(O/'laplace_curve/A_predictions.npy');old_mu=np.load(P/'outputs/controlled_experiments_v013/deep_curve/A_predictions.npy');scale=np.load(O/'laplace_curve/A_scale_predictions.npy')
assert abs(np.abs(mu-truth).mean()-new['validation_mae'])<1e-15
point_rc=np.broadcast_to(rc[val,None,:],(len(val),80,2))[mask];e=np.abs(mu-truth);oe=np.abs(old_mu-truth)
records=[]
for setting in sorted(set(map(tuple,rc[val]))):
 sel=np.all(point_rc==setting,axis=1)
 records.append(dict(R=int(setting[0]),C=int(setting[1]),points=int(sel.sum()),control_MAE=float(oe[sel].mean()),Laplace_MAE=float(e[sel].mean()),control_total_error=float(oe[sel].sum()),Laplace_total_error=float(e[sel].sum()),error_change=float((e[sel]-oe[sel]).sum()),Laplace_error_share=float(e[sel].sum()/e.sum()),scale_mean=float(scale[sel].mean())))
table=pd.DataFrame(records);table.to_csv(V/'R_C_comparison.csv',index=False)
print(table.to_string(index=False));print('R50 error share:',table.loc[table.R==50,'Laplace_total_error'].sum()/e.sum())
print('Late training diagnostic/validation:',new['epochs'][-1]['training_eval_inspiratory_mae'],new['epochs'][-1]['validation_inspiratory_mae'])


 R  C  points  control_MAE  Laplace_MAE  control_total_error  Laplace_total_error  error_change  Laplace_error_share  scale_mean
 5 10   22461     0.193912     0.181309          4355.453561          4072.375047   -283.078515             0.098733    0.089374
 5 20   23070     0.133455     0.127655          3078.814371          2945.003256   -133.811115             0.071400    0.074020
 5 50   22061     0.133976     0.136732          2955.645682          3016.452969     60.807287             0.073132    0.088547
20 10   16589     0.206780     0.190417          3430.267709          3158.820827   -271.446882             0.076584    0.102438
20 20   16704     0.182133     0.178006          3042.350745          2973.419374    -68.931372             0.072089    0.100849
20 50   21893     0.193024     0.196761          4225.883433          4307.677670     81.794236             0.104438    0.119319
50 10   37444     0.215099     0.209378          8054.166652          7839.954920   -214.211733  

In [3]:
fig,axes=plt.subplots(2,2,figsize=(13,9),constrained_layout=True)
for report,label,color in [(old,'Curve L1 control','#526d82'),(new,'Laplace curve','#d9792b')]:
 h=report['epochs'];epochs=[z['epoch'] for z in h];val_mae=[z['validation_inspiratory_mae'] for z in h]
 axes[0,0].plot(epochs,val_mae,label=label,color=color)
 late=[z for z in h if z['epoch']>=150]
 axes[0,1].plot([z['epoch'] for z in late],[z['validation_inspiratory_mae'] for z in late],label=label+' validation',color=color)
 axes[0,1].plot([z['epoch'] for z in late],[z['training_eval_inspiratory_mae'] for z in late],label=label+' training diagnostic',color=color,linestyle='--')
axes[0,0].set(title='Validation learning curves',xlabel='Epoch',ylabel='Inspiratory MAE');axes[0,0].set_yscale('log');axes[0,0].legend()
axes[0,1].set(title='Final 50 epochs: diagnostic fit gap remains',xlabel='Epoch',ylabel='Inspiratory MAE');axes[0,1].legend(fontsize=8)
x=np.arange(len(table));labels=[str(r)+' / '+str(c) for r,c in zip(table.R,table.C)]
axes[1,0].bar(x-.2,table.control_total_error,width=.4,label='Curve L1 control',color='#526d82');axes[1,0].bar(x+.2,table.Laplace_total_error,width=.4,label='Laplace curve',color='#d9792b');axes[1,0].set_xticks(x,labels,rotation=45,ha='right');axes[1,0].set(title='Total absolute error by lung setting',xlabel='R / C',ylabel='Total native absolute error');axes[1,0].legend()
axes[1,1].bar(x-.2,table.Laplace_MAE,width=.4,label='Actual mean absolute error',color='#d9792b');axes[1,1].bar(x+.2,table.scale_mean,width=.4,label='Mean predicted scale (not calibrated)',color='#79a4ad');axes[1,1].set_xticks(x,labels,rotation=45,ha='right');axes[1,1].set(title='Learned scale underestimates held-out error',xlabel='R / C',ylabel='Native pressure units');axes[1,1].legend(fontsize=8)
for ax in axes.flat:ax.grid(axis='y',alpha=.2)
fig.savefig(V/'learning_and_group_diagnostics.png',dpi=160);plt.close(fig)
print('Saved',V/'learning_and_group_diagnostics.png')


Saved /Users/generalclasher/Documents/dsc102/pa3/ventilator-pressure-prediction/outputs/controlled_experiments_v029/review/learning_and_group_diagnostics.png


The new objective improves the corresponding individual control while fitting its training diagnostic sample less closely. The gap to validation remains substantial. Learned scale is below actual held-out error, so it must not be presented as a calibrated confidence interval. R50 still contributes about half of total error.

The predeclared mixtures remain rejected: their negative bootstrap intervals do not override the practical0.003 gate. No epoch, scale or weight tuning was performed on confirmation/test outcomes. The saved-model V030 follow-on is a separate frozen experiment, not a retroactive V029 promotion.
